### Loading data.

In [1]:
import kagglehub
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [2]:
path = kagglehub.competition_download('filament-segmentation-2026')
print("Path to competition files:", path)

Path to competition files: /root/.cache/kagglehub/competitions/filament-segmentation-2026


In [3]:
def generate_mask_for_image(img_info, filaments):
    """
    It creates a binary mask (0 and 1).
    """
    height = img_info['height']
    width = img_info['width']

    # Background
    mask = np.zeros((height, width), dtype=np.uint8)

    for ann in filaments:
        segmentations = ann.get('segmentation', [])

        for seg in segmentations:
            poly = np.array(seg, dtype=np.int32).reshape((-1, 2))

            cv2.fillPoly(mask, [poly], color=1)

    return mask


In [4]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.utils import Sequence

class FilamentDataGenerator(Sequence):
    def __init__(self, images_list, annotations_map, img_dir, batch_size=8, target_size=(512, 512), shuffle=True):
        self.images_list = images_list
        self.annotations_map = annotations_map
        self.img_dir = img_dir
        self.batch_size = batch_size
        self.target_size = target_size
        self.shuffle = shuffle
        self.indexes = np.arange(len(self.images_list))
        self.on_epoch_end()

    def __len__(self):
      """
      Number of batch per epoch.
      """
      return int(np.ceil(len(self.images_list) / self.batch_size))

    def on_epoch_end(self):
      """
      Randomly shuffle the data for each epoch.
      """
      if self.shuffle:
          np.random.shuffle(self.indexes)

    def __getitem__(self, index):
      """
      Reads and resizes 8 photos and smoothes them.
      """

      batch_indexes = self.indexes[index * self.batch_size:(index + 1) * self.batch_size]

      batch_images = []
      batch_masks = []

      for idx in batch_indexes:
            img_info = self.images_list[idx]
            img_id = img_info['id']

            # Reading an image from disk.
            img_path = os.path.join(self.img_dir, img_info['file_name'])
            image = cv2.imread(img_path)
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

            # Generating a binary mask of the same image.
            filaments = self.annotations_map.get(img_id, [])
            mask = generate_mask_for_image(img_info, filaments)

            # On-the-fly resizing.
            image_resized = cv2.resize(image, self.target_size, interpolation=cv2.INTER_LINEAR)
            mask_resized = cv2.resize(mask, self.target_size, interpolation=cv2.INTER_NEAREST)

            # Data normalization (numbers between 0 and 1).
            image_normalized = image_resized.astype(np.float32)
            mask_normalized = np.expand_dims(mask_resized.astype(np.float32), axis=-1)

            batch_images.append(image_normalized)
            batch_masks.append(mask_normalized)

      return np.array(batch_images), np.array(batch_masks)

In [5]:
from sklearn.model_selection import train_test_split
import os
import json
import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Path
BASE_DIR = '/root/.cache/kagglehub/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026'
ANN_PATH = os.path.join(BASE_DIR, 'train', 'MAGFiLO_1.0_Annotations_kaggle2026_train.json')
# Reading json path
with open(ANN_PATH, 'r') as f:
    coco_data = json.load(f)
all_images = coco_data['images']

train_images, val_images = train_test_split(all_images, test_size=0.2, random_state=19)
IMG_DIR = os.path.join(BASE_DIR, 'train', 'train_images')

from collections import defaultdict

filaments_by_image = defaultdict(list)
for ann in coco_data['annotations']:
    filaments_by_image[ann['image_id']].append(ann)

train_generator = FilamentDataGenerator(
    images_list=train_images,
    annotations_map=filaments_by_image,
    img_dir=IMG_DIR,
    batch_size=8,
    target_size=(1024, 1024),
    shuffle=True
)

val_generator = FilamentDataGenerator(
    images_list=val_images,
    annotations_map=filaments_by_image,
    img_dir=IMG_DIR,
    batch_size=8,
    target_size=(1024, 1024),
    shuffle=False
)

print(f"Number of batch Train: {len(train_generator)}")
print(f"Number of batch Validation: {len(val_generator)}")

Number of batch Train: 116
Number of batch Validation: 29


In [6]:
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB0

encoder = EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(1024, 1024, 3)
)

encoder.trainable = True

for layer in encoder.layers[:-40]:
    layer.trainable = False

skip_names = [
    "block2a_expand_activation",
    "block3a_expand_activation",
    "block4a_expand_activation",
    "block6a_expand_activation",
    "top_activation"
]

skip_outputs = [
    encoder.get_layer(name).output
    for name in skip_names
]

encoder_model = Model(
    inputs=encoder.input,
    outputs=skip_outputs
)

In [7]:
def decoder_block(x, skip, filters):

    x = layers.Conv2DTranspose(
        filters,
        (2, 2),
        strides=(2, 2),
        padding="same"
    )(x)

    x = layers.Concatenate()([x, skip])

    x = layers.Conv2D(
        filters,
        (3, 3),
        padding="same",
        activation="relu"
    )(x)

    x = layers.BatchNormalization()(x)

    x = layers.Conv2D(
        filters,
        (3, 3),
        padding="same",
        activation="relu"
    )(x)

    x = layers.BatchNormalization()(x)

    x = layers.Conv2D(
        filters,
        (3, 3),
        padding="same",
        activation="relu"
    )(x)

    x = layers.BatchNormalization()(x)

    return x

In [8]:
def build_transfer_unet():

    inputs = layers.Input(shape=(1024, 1024, 3))

    skip1, skip2, skip3, skip4, skip5 = encoder_model(inputs)

    x = skip5

    x = decoder_block(x, skip4, 512)

    x = decoder_block(x, skip3, 256)

    x = decoder_block(x, skip2, 128)

    x = decoder_block(x, skip1, 64)

    x = layers.Conv2DTranspose(
        32,
        (2, 2),
        strides=(2, 2),
        padding="same"
    )(x)

    x = layers.Conv2D(
        32,
        (3, 3),
        padding="same",
        activation="relu"
    )(x)

    outputs = layers.Conv2D(
        1,
        (1, 1),
        activation="sigmoid"
    )(x)

    return Model(inputs, outputs)

In [9]:
import tensorflow.keras.backend as K

def dice_coef(y_true, y_pred, smooth=1e-4):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    dice = 1.0 - dice_coef(y_true, y_pred)
    return 0.3 * bce + 0.7 * dice

In [10]:
transfer_model = build_transfer_unet()

In [11]:
transfer_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=2e-4
    ),
    loss=bce_dice_loss,
    metrics=[dice_coef]
)

In [12]:
from tensorflow.keras.callbacks import ReduceLROnPlateau, ModelCheckpoint, EarlyStopping

callbacks = [
    ReduceLROnPlateau(monitor='val_dice_coef', factor=0.5, patience=3, min_lr=1e-6, mode='max', verbose=1),
    EarlyStopping(monitor='val_dice_coef', mode='max', patience=8, restore_best_weights=True, verbose=1),
    ModelCheckpoint('model_1024.keras', monitor='val_dice_coef', save_best_only=True, mode='max', verbose=1)
]

history_transfer = transfer_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    callbacks=callbacks
)

/usr/local/lib/python3.13/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20
116/116 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - dice_coef: 0.1057 - loss: 0.7108
Epoch 1: val_dice_coef improved from None to 0.00728, saving model to model_1024.keras

Epoch 1: finished saving model to model_1024.keras
116/116 ━━━━━━━━━━━━━━━━━━━━ 903s 5s/step - dice_coef: 0.3003 - loss: 0.5236 - val_dice_coef: 0.0073 - val_loss: 0.8050 - learning_rate: 2.0000e-04
Epoch 2/20
116/116 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - dice_coef: 0.6497 - loss: 0.2504
Epoch 2: val_dice_coef improved from 0.00728 to 0.49105, saving model to model_1024.keras

Epoch 2: finished saving model to model_1024.keras
116/116 ━━━━━━━━━━━━━━━━━━━━ 411s 4s/step - dice_coef: 0.6565 - loss: 0.2456 - val_dice_coef: 0.4910 - val_loss: 0.3618 - learning_rate: 2.0000e-04
Epoch 3/20
116/116 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - dice_coef: 0.6783 - loss: 0.2312
Epoch 3: val_dice_coef improved from 0.49105 to 0.64016, saving model to model_1024.keras

Epoch 3: finished saving model to model_1024.keras
116/116 ━━━━━━━━━━━━━━━

In [15]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from pycocotools import mask as mask_utils

transfer_model.load_weights('model_1024.keras')


def mask_to_counts(m):
    rle = mask_utils.encode(np.asfortranarray(m.astype(np.uint8)))
    return rle["counts"].decode("ascii")


TEST_DIR = os.path.join(BASE_DIR, 'test', 'test_images')

if not os.path.exists(TEST_DIR):
    TEST_DIR = os.path.join(BASE_DIR, 'test')

test_files = sorted([f for f in os.listdir(TEST_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

submission_data = []

for file_name in tqdm(test_files, desc="Predicting Test Set"):
    img_path = os.path.join(TEST_DIR, file_name)
    img_bgr = cv2.imread(img_path)

    if img_bgr is None:
        continue

    orig_h, orig_w = img_bgr.shape[:2]

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, (1024, 1024), interpolation=cv2.INTER_LINEAR).astype(np.float32)

    input_batch = np.expand_dims(img_resized, axis=0)

    pred = transfer_model.predict(input_batch, verbose=0)[0]

    prob_full = cv2.resize(pred[:, :, 0], (orig_w, orig_h), interpolation=cv2.INTER_LINEAR)
    binary = (prob_full > 0.45).astype(np.uint8)

    n, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    k = 0
    for i in range(1, n):
        if stats[i, cv2.CC_STAT_AREA] < 50:
            continue
        k += 1
        submission_data.append({
            'filament_id': f"{os.path.splitext(file_name)[0]}_{k}",
            'segmentation_rle': mask_to_counts(labels == i)
        })

Predicting Test Set: 100%|██████████| 180/180 [01:47<00:00,  1.67it/s]


In [16]:
df_submission = pd.DataFrame(submission_data)
df_submission.to_csv('submission.csv', index=False)
print(df_submission.shape)

(1705, 2)
